In [16]:
pip install requests beautifulsoup4 pandas vaderSentiment lxml undetected-chromedriver

Note: you may need to restart the kernel to use updated packages.


In [3]:
import re
import time
import pandas as pd
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [4]:
Products = {
    "iPhone 17 Pro Max": (
        "https://www.bestbuy.com/site/apple-iphone-17-pro-max-256gb-deep-blue-at-t/6473026.p",
        None,
    ),
    "Google Pixel 10 Pro": (
        "https://www.bestbuy.com/site/google-pixel-10-pro-xl-256gb-unlocked-obsidian/6637742.p",
        None,
    ),
    "Samsung Galaxy S26 Ultra": (
        "https://www.bestbuy.com/site/samsung-galaxy-s26-ultra-512gb-unlocked-black/6669749.p",
        "JJGRF36Y3Q",
    ),
}

In [5]:
def setting_driver() -> uc.Chrome:
    options=uc.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--disable-blink-features=AutomationControlled")
    return uc.Chrome(options=options,headless=False,version_main=146)

In [6]:
def extract_sku(url:str) -> str | None:
    m=re.search(r"/(\d{7,})", url)
    return m.group(1) if m else None

In [7]:
def extract_slug(url: str) -> str | None:
    m=re.search(r"/(?:site|product)/([^/]+)/", url)
    return m.group(1) if m else None

In [8]:
def discovering_url(driver: uc.Chrome, pdp_url: str) -> str | None:
    driver.get(pdp_url)
    WebDriverWait(driver, 30).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    time.sleep(5)
    current=driver.current_url
    m=re.search(r"(/product/[^/]+/[^/]+/sku/\d+)", current)
    if m:
        base="https://www.bestbuy.com"+m.group(1)
        return base + "/reviews?page={page}&pageSize=20"
    m=re.search(r"/product/([^/]+)/([A-Z0-9]{6,})\b", current)
    if m:
        slug, model=m.group(1), m.group(2)
        sku=extract_sku(pdp_url)
        if not sku:
            try:
                canonical= driver.find_element(
                    By.XPATH, "//link[@rel='canonical']"
                ).get_attribute("href")
                sku=extract_sku(canonical)
                if sku:
                    pass
            except Exception:
                pass
        if sku:
            base= f"https://www.bestbuy.com/product/{slug}/{model}/sku/{sku}"
            return base + "/reviews?page={page}&pageSize=20"
        else:
            base= f"https://www.bestbuy.com/product/{slug}/{model}"
            return base + "/reviews?page={page}&pageSize=20"
    try:
        link=driver.find_element(
            By.XPATH,
            "//a[contains(@href,'reviews') and not(contains(@href,'http'))]"
        )
        driver.execute_script("arguments[0].click();",link)
        time.sleep(4)
        current=driver.current_url
        m=re.search(r"(https://www\.bestbuy\.com/product/[^?#]+/sku/\d+)", current)
        if m:
            return m.group(1) + "/reviews?page={page}&pageSize=20"
    except Exception:
        pass
    return None

In [9]:
js_extract="""
const li=arguments[0];
let rating=null;
const srSpan=li.querySelector('span.sr-only');
if (srSpan) 
{
 const m=srSpan.innerText.match(/(\d+(?:\\.\\d+)?)\\s+out\\s+of\\s+5/i);
 if (m) rating=parseFloat(m[1]);
}
let title='';
const h4=li.querySelector('h4');
if (h4) title=h4.innerText.trim();
let body='';
li.querySelectorAll('p').forEach(p =>
{
    const cls=p.className || '';
    const tid=p.getAttribute('data-testid') || '';
    if (tid=='posted-by') return;
    if (cls.includes('disclaimer') || cls.includes('visually-hidden') || cls.includes('sr-only')) return;
    const txt=p.innerText.trim();
    if (txt.length > 20) body += (body ? ' ' : '') + txt;
    });
    let reviewer = '',date='';
    const pb=li.querySelector('[data-testid="posted-by"]');
    if (pb)
    {
        const txt=pb.innerText.trim();
        const m=txt.match(/^(.+?)\\s+Posted\\s+(.+)$/);
        if (m) { reviewer=m[1].trim(); date=m[2].trim(); }
        else { reviewer=txt; }
    }

    const badges=[];
    li.querySelectorAll('div.mb-200 span.v-text-tech-black').forEach( e1 => 
    {
     const t=e1.innerText.trim();
     if (t && !badges.includes(t)) badges.push(t);
    });
    return { rating,title,body,reviewer,date, badges: badges.join(' | ')};
    """

In [10]:
def parsing_js(driver, li_el) -> dict | None:
    try:
        r=driver.execute_script(js_extract, li_el)
        if not r["body"] and not r["title"]:
            return None
        return r
    except Exception as ex:
        print("\n Jaavascript error",ex)
        return None

In [11]:
def build_url(pdp_url: str, known_model: str) -> str | None:
    sku=extract_sku(pdp_url)
    slug=extract_slug(pdp_url)
    if sku and slug:
        return (
            f"https://www.bestbuy.com/product/{slug}/{known_model}"
            f"/sku/{sku}/reviews?page={{page}}&pageSize=20"
        )
    return None

In [12]:
def scrape(driver: uc.Chrome, product_name: str, pdp_url: str,
           known_model: str | None) -> list[dict]:
    wait=WebDriverWait(driver,30)
    url=None
    if known_model:
        url=build_url(pdp_url,known_model)
    if not url:
        url=discovering_url(driver,pdp_url)
        if not url:
            print(f"\n {product_name} url not found")
            return []
    print(f"\n {product_name} url found [{url}]")
    data=[]
    page=1
    while True:
        urll=url.format(page=page)
        driver.get(urll)
        print(f"\n Page number of {product_name} being scraped = {page}")
        try:
            wait.until(lambda d:d.execute_script("return document.readyState") == "complete")
        except:
            break
        try:
            WebDriverWait(driver,15).until(
                EC.presence_of_element_located((By.ID, "stand-alone-review-list"))
            )
        except Exception:
            time.sleep(4)
        time.sleep(3)
        try:
            ul=driver.find_element(By.ID, "stand-alone-review-list")
            items=ul.find_elements(By.XPATH, "./li")
            print(f" \n {product_name} has {len(items)} reviews on page number {page}")
        except Exception:
            break
        if not items:
            break
        for li in items:
            parsed=parsing_js(driver,li)
            if parsed:
                data.append({
                    "Mobile_Device": product_name,
                    "Rating": parsed["rating"],
                    "Title": parsed["title"],
                    "Review": parsed["body"],
                    "Reviewer_Name": parsed["reviewer"],
                    "Date": parsed["date"],
                    "Additional_Information": parsed["badges"],
                })
            #print(f"\n Number of reviews collected for {product_name} until now {len(data)}")
            if parsed is None:
                print("\n Parsing Failed")
        if len(items) < 20:
            break
        page+=1
        time.sleep(3)
    return data

In [13]:
def main():
    driver=setting_driver()
    dataa=[]
    try:
        for product_name,(pdp_url, known_model) in Products.items():
            print(f"\n Scrapping: {product_name} ")
            reviews=scrape(driver,product_name,pdp_url,known_model)
            print(f"\n Total Reviews found for {product_name} = {len(reviews)}")
            dataa.extend(reviews)
            time.sleep(5)
    finally:
        driver.quit()
    print(f"\n Total reviews scraped = {len(dataa)}")
    if dataa:
        df=pd.DataFrame(dataa)
        df.to_csv("Reviews.csv", index=False)
        print("\n CSV file created")
    else:
        print("\n CSV file creation failed")
if __name__ =="__main__":
    main()


 Scrapping: iPhone 17 Pro Max 

 iPhone 17 Pro Max url found [https://www.bestbuy.com/product/apple-iphone-17-pro-max-256gb-deep-blue-at-t/JCQ6HQTWP9/sku/6473026/reviews?page={page}&pageSize=20]

 Page number of iPhone 17 Pro Max being scraped = 1
 
 iPhone 17 Pro Max has 20 reviews on page number 1

 Page number of iPhone 17 Pro Max being scraped = 2
 
 iPhone 17 Pro Max has 20 reviews on page number 2

 Page number of iPhone 17 Pro Max being scraped = 3
 
 iPhone 17 Pro Max has 20 reviews on page number 3

 Page number of iPhone 17 Pro Max being scraped = 4
 
 iPhone 17 Pro Max has 20 reviews on page number 4

 Page number of iPhone 17 Pro Max being scraped = 5
 
 iPhone 17 Pro Max has 20 reviews on page number 5

 Page number of iPhone 17 Pro Max being scraped = 6
 
 iPhone 17 Pro Max has 20 reviews on page number 6

 Page number of iPhone 17 Pro Max being scraped = 7
 
 iPhone 17 Pro Max has 20 reviews on page number 7

 Page number of iPhone 17 Pro Max being scraped = 8
 
 iPhone